In [22]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [23]:

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment_name = "gpt-5.4-mini"

openai_client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)


In [24]:
def llm(prompt):
    response = openai_client.responses.create(
        model=deployment_name,
        input=prompt,
    )
    return response.output

In [25]:

response = llm("can i still join the course ?")

response

[ResponseOutputMessage(id='msg_0b518de116ecce33006a099acc391c8193bc49d546559eba1f', content=[ResponseOutputText(annotations=[], text='Yes, probably — but it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you check or draft a message. A simple version is:\n\n“Hi, I’m interested in joining the course. Is it still possible to enroll?”\n\nIf you’d like, I can also make it more formal, friendly, or specific to a school/training program.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [5]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''


In [ ]:
question = "are you single ?"

In [7]:
prompt = f"""you are an assistant for a course, assist students by answering their questions based on the context provided. 
Say 'oops, i dont know' if the answer is not in the context. Always use the context to answer, do not use any outside information. 
Context: {context} 
Question: {question}"""

In [8]:
answer = llm(prompt)
print(answer)

[ResponseOutputMessage(id='msg_0cd615b0423206cf006a05fb52766c81949ecf9881d90cf731', content=[ResponseOutputText(annotations=[], text='oops, i dont know', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]


In [1]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [2]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [4]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [3]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [23]:
import pandas as pd
from pprint import pprint
df = pd.DataFrame(documents)

pprint(df['course'].unique().tolist())

['machine-learning-zoomcamp',
 'llm-zoomcamp',
 'data-engineering-zoomcamp',
 'mlops-zoomcamp']


In [4]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [15]:
question = "can i join the course now?"
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have r

In [17]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()


In [18]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''


USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''


In [19]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()


In [ ]:
prompt = build_prompt(question, search_results)


In [ ]:

print(prompt)


In [26]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)

In [30]:
print(response.output_text)


Yes — you can still join now.

You can start learning and submit homework while the submission form is still open, even if you didn’t register earlier. If you want a certificate, though, you must finish with the live cohort and submit your project while submissions are still being accepted.


In [ ]:
response.usage

ResponseUsage(input_tokens=330, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=61, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=391)

In [33]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text


In [34]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [35]:
answer = rag('I just discovered the course. Can I join now?')
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
